# REATS — Real-time Explainable ATR · Kaggle T4 Full Pipeline

**Data Ingestion → Multi-Viewpoint Aug → Module B Train → Module C XAI → End-to-End Demo**

| Module | Role | Target |
|--------|------|--------|
| **A** | YOLOv4 detector (CSPDarknet53 + SPP + PANet) | mAP@0.5 ≥ 75% |
| **B** | Heterogeneous 6-architecture ensemble (ConvNeXt_tiny, ResNeXt50, ViT_b_16, Swin_T, VGG16, ResNet18) | Accuracy ≥ 92 %, ECE ≤ 0.05 |
| **C** | GradCAM / SHAP / MC-Dropout XAI | Faithfulness AUC ≥ 0.80 |
| **D** | Streamlit live dashboard | ≤ 40 ms/frame, ≥ 20 FPS |

**Taxonomy**: 43 military targets — AIR (24 classes) · GROUND (13) · NAVAL (6)

**Before running**: Edit → Notebook settings → Accelerator → **GPU T4 x1**

---

### Kaggle inputs to add (right panel → + Add input)

| Input | Pipeline key | Domain |
|---|---|---|
| `deepnewbie/flir-thermal-images-dataset` | `FLIR_Thermal` | thermal IR |
| `samdazel/teledyne-flir-adas-thermal-dataset-v2` | `FLIR_ADAS_v2` | thermal IR |
| `pandrii000/hituav-a-highaltitude-infrared-thermal-dataset` | `HIT_UAV` | thermal aerial |
| `trnhvtunt/dataset1` (HIT-UAV v1.2.1) | `HIT_UAV_v2` | thermal aerial |
| `trnhvtunt/dataset2` (IR mp4 clips) | `Dataset2_Folders` | air |
| `guofeng/hrsc2016` | `HRSC2016` | naval |
| `andrewmvd/ship-detection` | `Ships_Aerial` | naval |
| `tomluther/ships-in-google-earth` | `Ships_Google_Earth` | naval |
| `siddharthkumarsah/ships-in-aerial-images` | `Ships_Vessels_Aerial` | naval |
| `lilitopia/swimship-wake-imagery-mass` | `SWIM` | naval |
| `airbusgeo/airbus-aircrafts-sample-dataset` | `CGI_Planes` | air |
| `kbhartiya83/swimming-pool-and-car-detection` | `SwimmingPool_Car` | ground |
| `alpereniek/vehicle-detection-from-satellite-images-data-set` | `Vehicle_Dataset` | ground |
| `humansintheloop/semantic-segmentation-of-aerial-imagery` | `Aerial_Segmentation` | mixed |
| `atilol/aerialimageryforroofsegmentation` | `Aerial_Roof_Seg` | (null labels) |
| notebook `trnhvtunt/real-time-ex-01` | warm-start checkpoints | — |

> Any missing input is silently skipped — the synthetic fallback fills the gap automatically.

In [1]:
import torch, sys, platform

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'Platform: {platform.platform()}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    p = torch.cuda.get_device_properties(0)
    vram_gb = p.total_memory / 1e9
    print(f'\nGPU  : {p.name}')
    print(f'VRAM : {vram_gb:.1f} GB')
    BATCH_SIZE = 128 if vram_gb >= 20 else 64   # A100 → 128, T4 → 64
else:
    print('WARNING: No GPU — enable in Notebook settings → Accelerator')
    BATCH_SIZE = 16

print(f'Auto batch_size = {BATCH_SIZE}')

Python  : 3.12.13
PyTorch : 2.11.0+cpu
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Auto batch_size = 16


In [2]:
import subprocess, sys

deps = [
    'kornia>=0.7.0',
    'grad-cam>=1.4.8',
    'shap>=0.44.0',
    'mlflow>=2.10.0',
    'torchmetrics>=1.0.0',
    'pyyaml>=6.0',
    'seaborn>=0.12.0',
    'lime>=0.2.0.1',
    'pyngrok>=7.0',
    'opencv-python-headless>=4.8.0',
    'streamlit>=1.30.0',
    'watchdog>=3.0.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + deps)
print('All packages installed.')

All packages installed.


In [3]:
import importlib, shutil, subprocess, sys, time
from pathlib import Path

REPO   = 'https://github.com/trinhvutuandat34/Real-time-Explainable-Automatic-Target-Recognition-System.git'
BRANCH = 'main'
ROOT   = Path('/kaggle/working/reats')
REATS  = ROOT / 'REATS'

if ROOT.exists():
    r = subprocess.run(
        ['git', '-C', str(ROOT), 'pull', 'origin', BRANCH],
        capture_output=True, text=True,
    )
    print(r.stdout.strip() or 'Already up to date.')
else:
    subprocess.check_call(
        ['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, str(ROOT)]
    )
    print(f'Cloned {BRANCH} -> {ROOT}')

# Clear stale .pyc caches so git-pulled changes take effect immediately
for _cache in ROOT.rglob('__pycache__'):
    shutil.rmtree(_cache, ignore_errors=True)
for _mod in [k for k in sys.modules if k.split('.')[0] in ('config', 'ingestion', 'modules')]:
    del sys.modules[_mod]
importlib.invalidate_caches()

for p in [str(REATS), str(ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Fallback if c-gpu (cell 1) hasn't been run yet
try:
    device
except NameError:
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

from config import CLASSES, NUM_CLASSES, TARGET_META
from modules.module_b_classifier import (
    build_convnext, EnsembleClassifier, build_loaders,
    train_full_pipeline, train_ensemble, load_ensemble,
    evaluate, compute_ece, KorniaAugmentPipeline,
)
from modules.module_c_xai import (
    GradCAMExplainer, MCDropoutWrapper,
    faithfulness_deletion_auc, faithfulness_insertion_auc,
    overlay_heatmap,
)
from modules.augmentation_viewpoint import MultiViewpointAugmentor
from ingestion.pipeline import IngestPipeline

print(f'REATS loaded OK | {NUM_CLASSES} classes | device={device}')
print('First 8 classes:', CLASSES[:8])

Cloned main -> /kaggle/working/reats
REATS loaded OK | 43 classes | device=cpu
First 8 classes: ['F16', 'F15', 'F22', 'F35', 'Su27', 'Su35', 'MiG29', 'MiG19']


In [4]:
import shutil
from pathlib import Path

# ── Output paths ────────────────────────────────────────────────────────
DATA_DIR = Path('/kaggle/working/reats/REATS/data')
CKPT_DIR = Path('/kaggle/working/checkpoints')
RUNS_DIR = Path('/kaggle/working/mlruns')
for d in [DATA_DIR, CKPT_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Single-model checkpoint — defined here (not in the training cell) so the
# evaluation cells also work in resumed sessions that skip training.
CKPT_SINGLE = str(CKPT_DIR / 'convnext_best.pth')

# ── Warm start from the previous run's output (mounted as an input) ────
# Add via right panel → + Add input → Notebooks → trnhvtunt/real-time-ex-01.
# Its checkpoints let you skip straight to evaluation / calibration /
# dashboard cells without the ~4 h retrain.
PREV_RUN = Path('/kaggle/input/notebooks/trnhvtunt/real-time-ex-01')
if PREV_RUN.exists():
    copied = []
    for pat in ('*.pth', '*.pt', 'temperature.json'):
        for src in PREV_RUN.rglob(pat):
            dst = CKPT_DIR / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                copied.append(src.name)
    print('Warm start from previous run: '
          + (', '.join(copied) if copied else 'nothing new to copy'))
else:
    print('No previous-run mount — training cells will produce fresh checkpoints.')

# ── Kaggle input dataset paths ──────────────────────────────────────────
# Keys MUST match entries in ingestion/label_maps.yaml exactly.
# Any path that does not exist is silently skipped; synthetic fallback
# fills the remaining shortfall automatically.
#
# Mounted datasets (right panel → + Add input):
#   deepnewbie/flir-thermal-images-dataset            → FLIR_Thermal
#   samdazel/teledyne-flir-adas-thermal-dataset-v2    → FLIR_ADAS_v2
#   pandrii000/hituav-a-highaltitude-infrared-thermal-dataset → HIT_UAV
#   trnhvtunt/dataset1  (HIT-UAV v1.2.1 COCO/XML + Images/Images) → HIT_UAV_v2
#   trnhvtunt/dataset2  (fixed_wing / rotary_wing mp4 clips)      → Dataset2_Folders
#   guofeng/hrsc2016                                  → HRSC2016
#   andrewmvd/ship-detection                          → Ships_Aerial
#   tomluther/ships-in-google-earth                   → Ships_Google_Earth
#   siddharthkumarsah/ships-in-aerial-images          → Ships_Vessels_Aerial
#   lilitopia/swimship-wake-imagery-mass              → SWIM
#   organizations/airbusgeo/airbus-aircrafts-sample-dataset → CGI_Planes
#   kbhartiya83/swimming-pool-and-car-detection       → SwimmingPool_Car
#   alpereniek/vehicle-detection-from-satellite-images-data-set → Vehicle_Dataset
#   humansintheloop/semantic-segmentation-of-aerial-imagery → Aerial_Segmentation
#   atilol/aerialimageryforroofsegmentation           → Aerial_Roof_Seg
DATASET_INPUTS = {
    # ── Thermal IR ──────────────────────────────────────────────────────
    'FLIR_Thermal':         Path('/kaggle/input/datasets/deepnewbie/flir-thermal-images-dataset'),
    'FLIR_ADAS_v2':         Path('/kaggle/input/datasets/samdazel/teledyne-flir-adas-thermal-dataset-v2'),
    'HIT_UAV':              Path('/kaggle/input/datasets/pandrii000/hituav-a-highaltitude-infrared-thermal-dataset'),
    # ── Naval ────────────────────────────────────────────────────────────
    'HRSC2016':             Path('/kaggle/input/datasets/guofeng/hrsc2016'),
    'Ships_Aerial':         Path('/kaggle/input/datasets/andrewmvd/ship-detection'),
    'Ships_Google_Earth':   Path('/kaggle/input/datasets/tomluther/ships-in-google-earth'),
    'Ships_Vessels_Aerial': Path('/kaggle/input/datasets/siddharthkumarsah/ships-in-aerial-images'),
    'SWIM':                 Path('/kaggle/input/datasets/lilitopia/swimship-wake-imagery-mass'),
    # ── Air ──────────────────────────────────────────────────────────────
    'CGI_Planes':           Path('/kaggle/input/datasets/organizations/airbusgeo/airbus-aircrafts-sample-dataset'),
    # ── Ground ───────────────────────────────────────────────────────────
    'SwimmingPool_Car':     Path('/kaggle/input/datasets/kbhartiya83/swimming-pool-and-car-detection'),
    'Vehicle_Dataset':      Path('/kaggle/input/datasets/alpereniek/vehicle-detection-from-satellite-images-data-set'),
    # ── Mixed aerial ─────────────────────────────────────────────────────
    'Aerial_Segmentation':  Path('/kaggle/input/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery'),
    # ── User datasets ────────────────────────────────────────────────────
    'HIT_UAV_v2':           Path('/kaggle/input/datasets/trnhvtunt/dataset1'),
    'Dataset2_Folders':     Path('/kaggle/input/datasets/trnhvtunt/dataset2'),
    'Aerial_Roof_Seg':      Path('/kaggle/input/datasets/atilol/aerialimageryforroofsegmentation'),
}

SPLIT_TARGETS = {'train': 170, 'val': 30, 'test': 200}   # per-class, per paper

# ── Module B hyperparams ────────────────────────────────────────────────
CONFIG = {
    'data_root':        str(DATA_DIR),
    'num_classes':      NUM_CLASSES,
    'img_size':         224,
    'batch_size':       BATCH_SIZE,
    'lr':               1e-4,
    'epochs':           300,
    'best_epoch_start': 225,   # checkpoint only from epoch 225, per paper
    'device':           device,
    'classes':          CLASSES,
    'warmup_epochs':    10,
    'min_lr':           1e-6,
    'weight_decay':     1e-2,
    'grad_clip':        1.0,
    'label_smoothing':  0.1,
    'ema_decay':        0.9999,
}

available_ds = [k for k, p in DATASET_INPUTS.items() if p.exists()]
missing_ds   = [k for k in DATASET_INPUTS if k not in available_ds]
print(f'Available Kaggle datasets ({len(available_ds)}/{len(DATASET_INPUTS)}): {available_ds}')
if missing_ds:
    print(f'Missing (skipped): {missing_ds}')
print(f'Classes: {NUM_CLASSES} | Batch: {BATCH_SIZE} | Epochs: {CONFIG["epochs"]}')

No previous-run mount — training cells will produce fresh checkpoints.
Available Kaggle datasets (0/15): []
Missing (skipped): ['FLIR_Thermal', 'FLIR_ADAS_v2', 'HIT_UAV', 'HRSC2016', 'Ships_Aerial', 'Ships_Google_Earth', 'Ships_Vessels_Aerial', 'SWIM', 'CGI_Planes', 'SwimmingPool_Car', 'Vehicle_Dataset', 'Aerial_Segmentation', 'HIT_UAV_v2', 'Dataset2_Folders', 'Aerial_Roof_Seg']
Classes: 43 | Batch: 16 | Epochs: 300


## Section 1 — Dataset

Two-stage pipeline:
1. **`IngestPipeline`** — parses all available Kaggle datasets (COCO JSON / YOLO txt / Pascal VOC XML / CSV / folder), maps source labels to the 43-class REATS taxonomy via `label_maps.yaml`, extracts 224×224 IR patches.
2. **`generate_flir_fallback.py`** — fills any remaining shortfall toward 170/30/200 images per class. Uses real FLIR ADAS bboxes if available; falls back to synthetic IR blob generation.

Both steps are idempotent — re-runs only fill the gap.

In [5]:
datasets_available = {k: v for k, v in DATASET_INPUTS.items() if v.exists()}

if datasets_available:
    print(f'Running IngestPipeline on {len(datasets_available)} dataset(s)...')
    pipe = IngestPipeline(
        datasets      = datasets_available,
        out_root      = DATA_DIR,
        split_targets = SPLIT_TARGETS,
        seed          = 42,
    )
    written = pipe.run()
    total_written = sum(n for split in written.values() for n in split.values())
    print(f'Ingestion complete: {total_written} patches written.')
else:
    print('No Kaggle datasets found — will use synthetic fallback only.')
    print('Tip: add datasets via right panel + Add data for higher accuracy.')

No Kaggle datasets found — will use synthetic fallback only.
Tip: add datasets via right panel + Add data for higher accuracy.


In [6]:
import subprocess

FALLBACK = REATS / 'generate_flir_fallback.py'
flir_path = DATASET_INPUTS.get('FLIR_Thermal')

cmd = ['python', str(FALLBACK), '--out', str(DATA_DIR)]
if flir_path and flir_path.exists():
    cmd += ['--flir', str(flir_path)]
    print('Fallback: using FLIR ADAS crops + synthetic for remaining shortfall...')
else:
    print('Fallback: synthetic-only (no FLIR dataset found)...')

r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REATS))
if r.stdout:
    print(r.stdout[-3000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])

# Validate the final split counts
r2 = subprocess.run(
    ['python', str(REATS / 'dataset_validator.py'), '--root', str(DATA_DIR)],
    capture_output=True, text=True, cwd=str(REATS),
)
print(r2.stdout)

Fallback: synthetic-only (no FLIR dataset found)...
22M: Generated 30/30
  [bg ] val/Tu95: Generated 30/30
  [bg ] val/AH64: Generated 30/30
  [bg ] val/Mi24: Generated 30/30
  [bg ] val/Ka52: Generated 30/30
  [bg ] val/LYNX: Generated 30/30
  [bg ] val/UH60: Generated 30/30
  [bg ] val/CH47: Generated 30/30
  [bg ] val/MQ9: Generated 30/30
  [bg ] val/TB2: Generated 30/30
  [bg ] val/Shahed136: Generated 30/30
  [bg ] val/RQ4: Generated 30/30
  [bg ] val/WZ7: Generated 30/30
  [bg ] val/M1Abrams: Generated 30/30
  [bg ] val/T72: Generated 30/30
  [bg ] val/T90: Generated 30/30
  [bg ] val/Leopard2: Generated 30/30
  [bg ] val/BMP2: Generated 30/30
  [bg ] val/Bradley: Generated 30/30
  [bg ] val/BTR80: Generated 30/30
  [bg ] val/K21: Generated 30/30
  [bg ] val/M109: Generated 30/30
  [bg ] val/BM21: Generated 30/30
  [bg ] val/Patriot: Generated 30/30
  [bg ] val/Buk: Generated 30/30
  [bg ] val/Pantsir: Generated 30/30
  [bg ] val/PKG: Generated 30/30
  [bg ] val/PTG: Generated 30

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# One sample per domain row, 5 columns
DOMAINS = ['AIR', 'GROUND', 'NAVAL']
COLS    = 5
domain_samples = {d: [] for d in DOMAINS}

for cls in CLASSES:
    dom = TARGET_META[cls]['domain']
    if dom in domain_samples and len(domain_samples[dom]) < COLS:
        img_dir = DATA_DIR / 'train' / cls
        imgs = sorted(img_dir.glob('*'))[:1] if img_dir.exists() else []
        if imgs:
            domain_samples[dom].append((cls, imgs[0]))

dom_color = {'AIR': '#1565C0', 'GROUND': '#2E7D32', 'NAVAL': '#00695C'}

fig, axes = plt.subplots(3, COLS, figsize=(COLS * 3, 9))
for row, dom in enumerate(DOMAINS):
    for col in range(COLS):
        ax = axes[row, col]
        if col < len(domain_samples[dom]):
            cls_id, img_path = domain_samples[dom][col]
            meta = TARGET_META[cls_id]
            try:
                ax.imshow(Image.open(img_path), cmap='inferno')
                ax.set_title(f"{meta['name']}\n[{cls_id}]", fontsize=7,
                             color=dom_color[dom], pad=2)
            except Exception:
                ax.set_title(cls_id, fontsize=7)
        else:
            ax.set_facecolor('#111')
            ax.text(0.5, 0.5, 'n/a', ha='center', va='center',
                    transform=ax.transAxes, color='gray')
        ax.axis('off')
    axes[row, 0].set_ylabel(dom, fontsize=13, fontweight='bold',
                             color=dom_color[dom], labelpad=6)

plt.suptitle('Sample IR patches by domain (inferno = simulated thermal palette)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/sample_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_grid.png')

In [ ]:
import matplotlib.pyplot as plt

split_counts = {'train': [], 'val': [], 'test': []}
for split in split_counts:
    for cls in CLASSES:
        d = DATA_DIR / split / cls
        split_counts[split].append(sum(1 for _ in d.glob('*')) if d.exists() else 0)

x     = range(NUM_CLASSES)
w     = 0.28
fig, ax = plt.subplots(figsize=(max(14, NUM_CLASSES * 0.45), 5))
ax.bar([i - w for i in x], split_counts['train'], w, label='train', color='#1976D2', alpha=0.8)
ax.bar(x,                   split_counts['val'],   w, label='val',   color='#388E3C', alpha=0.8)
ax.bar([i + w for i in x], split_counts['test'],  w, label='test',  color='#F57C00', alpha=0.8)
ax.axhline(170, ls='--', color='#1976D2', lw=0.8, alpha=0.5, label='train target')
ax.axhline(200, ls='--', color='#F57C00', lw=0.8, alpha=0.5, label='test target')
ax.set_xticks(list(x))
ax.set_xticklabels(CLASSES, rotation=90, fontsize=7)
ax.set_ylabel('Images')
tr = sum(split_counts['train'])
va = sum(split_counts['val'])
te = sum(split_counts['test'])
ax.set_title(f'Dataset — {NUM_CLASSES} classes  ({tr} train / {va} val / {te} test)')
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=120)
plt.show()
print('Saved: class_distribution.png')

## Section 2 — Module B: ConvNeXt Training

`train_full_pipeline` runs the complete training loop with:
- **Augmentation**: `MultiViewpointAugmentor` (5 physics-driven UAV viewpoint transforms) → `KorniaAugmentPipeline` (10 standard IR augmentations) — both wired in automatically
- **AMP** (automatic mixed precision) on CUDA
- **EMA** (exponential moving average, decay=0.9999) — used for validation/checkpointing
- **Warmup-cosine** LR scheduler (10 warm-up epochs → cosine decay to 1e-6)
- **MLflow** run logged to `/kaggle/working/mlruns/mlflow.db`
- Checkpoint saved whenever val_acc improves, from epoch 225 onward (per paper)

In [ ]:
import mlflow, time

mlflow.set_tracking_uri(f'sqlite:///{RUNS_DIR}/mlflow.db')

print(f'Training ConvNeXt_tiny  ({CONFIG["epochs"]} epochs, best from epoch {CONFIG["best_epoch_start"]})')
print(f'Augmentor: MultiViewpointAugmentor + KorniaAugmentPipeline')
print(f'AMP + EMA + WarmupCosine | checkpoint -> {CKPT_SINGLE}')
print()

t0 = time.time()
best_val_acc, ckpt_path = train_full_pipeline(CONFIG, seed=42, ckpt_path=CKPT_SINGLE)
elapsed_min = (time.time() - t0) / 60

if best_val_acc >= 0.92:
    status = 'PASS'
elif best_val_acc >= 0.90:
    status = 'CLOSE  -> run ensemble cell below'
else:
    status = 'FAIL   -> check data, run ensemble cell below'

print(f'Best val acc : {best_val_acc:.4f}  [{status}]  (target >= 0.92)')
print(f'Wall time    : {elapsed_min:.1f} min')

In [ ]:
# ── OPTIONAL — run if single model is below 0.92 ──────────────────────
# Trains the 6 heterogeneous architectures required by Do et al. (2025) —
# ConvNeXt_tiny, ResNeXt50_32x4d, ViT_b_16, Swin_T, VGG16, ResNet18, one
# model per architecture (not 6 seeds of one) — then softmax-averages them.
# Time: ~6x longer than the single-model cell.
# Each model checkpointed separately; session restarts are safe to resume.

from modules.module_b_classifier import ARCHITECTURES

# Resume-safe: if c-train-single was skipped this session, recover
# best_val_acc from the existing checkpoint (training stores it there).
try:
    best_val_acc
except NameError:
    import torch
    from pathlib import Path
    best_val_acc = 0.0
    if Path(CKPT_SINGLE).exists():
        _ck = torch.load(CKPT_SINGLE, map_location='cpu')
        best_val_acc = float(_ck.get('best_val_acc', 0.0)) if isinstance(_ck, dict) else 0.0
        print(f'Resumed best_val_acc={best_val_acc:.4f} from {CKPT_SINGLE}')

TRAIN_ENSEMBLE = best_val_acc < 0.92

if TRAIN_ENSEMBLE:
    print(f'Single model {best_val_acc:.4f} < 0.92 — training heterogeneous '
          f'ensemble: {ARCHITECTURES}')
    ckpt_paths = train_ensemble(CONFIG, ckpt_dir=str(CKPT_DIR))
    ensemble   = load_ensemble(ckpt_paths, num_classes=NUM_CLASSES, device=device)
    ensemble.eval()
    USE_ENSEMBLE = True
    print(f'Ensemble ready: {len(ckpt_paths)} models')
else:
    print(f'Single model reached {best_val_acc:.4f} >= 0.92 — ensemble not needed.')
    USE_ENSEMBLE = False

## Section 3 — Evaluation

In [ ]:
import torch
import torch.nn as nn
from collections import defaultdict

# Resume-safe: if the training cells were skipped this session, detect an
# existing ensemble on disk; otherwise fall back to the single checkpoint.
try:
    USE_ENSEMBLE
except NameError:
    from modules.module_b_classifier import ARCHITECTURES
    _ens_paths = [CKPT_DIR / f'{arch}_{i}.pth' for i, arch in enumerate(ARCHITECTURES)]
    USE_ENSEMBLE = all(p.exists() for p in _ens_paths)
    if USE_ENSEMBLE:
        ensemble = load_ensemble([str(p) for p in _ens_paths], num_classes=NUM_CLASSES, device=device)
        ensemble.eval()
        print(f'Resumed {len(ARCHITECTURES)}-architecture heterogeneous ensemble from disk.')

# Load the best model (or ensemble) for evaluation
if USE_ENSEMBLE:
    eval_model = ensemble
else:
    eval_model = build_convnext(NUM_CLASSES).to(device)
    ckpt  = torch.load(CKPT_SINGLE, map_location=device)
    state = ckpt.get('ema_state_dict', ckpt.get('state_dict', ckpt))
    eval_model.load_state_dict(state)
eval_model.eval()

_, val_loader, test_loader = build_loaders(CONFIG)
criterion = nn.CrossEntropyLoss()

_, test_acc = evaluate(eval_model, test_loader, criterion, device)
ece = compute_ece(eval_model, test_loader, device, is_probs=USE_ENSEMBLE)

chk = lambda v, t, op='ge': ('PASS' if (v >= t if op == 'ge' else v <= t) else 'FAIL')

print('=' * 52)
print(f'  Test Accuracy : {test_acc:.4f}   [{chk(test_acc, 0.92)}]  target >= 0.92')
print(f'  ECE           : {ece:.5f}  [{chk(ece, 0.05, "le")}]  target <= 0.05')
print(f'  Model type    : {"Heterogeneous 6-architecture ensemble" if USE_ENSEMBLE else "Single ConvNeXt_tiny"}')
print('=' * 52)

# Collect all predictions for downstream analysis
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = eval_model(imgs.to(device))
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())

# ImageFolder assigns indices alphabetically; recover name -> idx mapping
idx_to_cls  = {v: k for k, v in test_loader.dataset.class_to_idx.items()}
ordered_cls = test_loader.dataset.classes   # alphabetically sorted for cm axes

# Per-domain accuracy breakdown
dom_correct = defaultdict(int)
dom_total   = defaultdict(int)
for pred, true in zip(all_preds, all_labels):
    cls_id = idx_to_cls.get(true, '')
    dom    = TARGET_META.get(cls_id, {}).get('domain', 'UNKNOWN')
    dom_correct[dom] += int(pred == true)
    dom_total[dom]   += 1

print()
print('Per-domain accuracy:')
for dom in ['AIR', 'GROUND', 'NAVAL']:
    tot = dom_total.get(dom, 0)
    cor = dom_correct.get(dom, 0)
    acc = cor / tot if tot else 0.0
    print(f'  {dom:<8}  {acc:.4f}  ({cor}/{tot})')

### Battlefield threat analysis — False Alarm Rate / Miss Rate

Beyond the paper's accuracy/ECE targets: **FAR** (`1 - Precision`) is the rate a civilian/friendly-like target gets misidentified as this threat class; **MR** (`1 - Recall`) is the rate an actual instance of the class is never flagged — the critical figure for RED-threat classes, since a missed enemy fighter never raises an alarm. No paper/professor-given numeric target exists for these; they're reported alongside the paper's metrics, not pass/fail gated. Reuses the `all_preds`/`all_labels` already collected above — no extra inference.

In [ ]:
from modules.threat_metrics import compute_far_mr, format_report

far_mr = compute_far_mr(all_labels, all_preds)
print(format_report(far_mr, top_n=10))
print()
print(f"Macro FAR: {far_mr['macro_FAR']:.4f}   Macro MR: {far_mr['macro_MR']:.4f}")
print(f"RED-threat FAR: {far_mr['red_threat_FAR']:.4f}   "
      f"RED-threat MR: {far_mr['red_threat_MR']:.4f}  "
      f"({far_mr['n_red_threats']} RED classes — missed-fighter risk)")

### Temperature scaling (post-hoc calibration)

Raw ECE overshoots the ≤ 0.05 target because the model is accurate but **over-confident**. A single scalar temperature `T`, fit on the validation set by minimising NLL (Guo et al. 2017), rescales the logits to restore calibration **without changing any prediction** — accuracy is untouched. The fitted `T` is saved to `checkpoints/temperature.json`; the dashboard Calibration tab loads it automatically as its slider default.

In [ ]:
import torch, torch.nn as nn, numpy as np, json

def _collect_logits(model, loader):
    """Return (logits, labels) on CPU. EnsembleClassifier emits probabilities,
    so convert those to log-probs to get a logit-space quantity to scale."""
    lg_all, lb_all = [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels in loader:
            out = model(imgs.to(device))
            lg  = torch.log(out.clamp_min(1e-8)) if USE_ENSEMBLE else out
            lg_all.append(lg.cpu()); lb_all.append(labels)
    return torch.cat(lg_all), torch.cat(lb_all)

def _ece_from_logits(logits, labels, T=1.0, n_bins=15):
    """ECE with the same binning as modules.module_b_classifier.compute_ece."""
    probs      = torch.softmax(logits / T, dim=-1)
    conf, pred = probs.max(dim=-1)
    conf       = conf.numpy()
    correct    = (pred.numpy() == labels.numpy()).astype(float)
    bins, e, n = np.linspace(0, 1, n_bins + 1), 0.0, len(conf)
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            e += abs(correct[m].mean() - conf[m].mean()) * m.sum() / n
    return float(e)

# Fit T on val (never on test), then evaluate calibration on test
val_logits,  val_labels  = _collect_logits(eval_model, val_loader)
test_logits, test_labels = _collect_logits(eval_model, test_loader)

T   = torch.ones(1, requires_grad=True)
opt = torch.optim.LBFGS([T], lr=0.01, max_iter=100)
nll = nn.CrossEntropyLoss()

def _closure():
    opt.zero_grad()
    loss = nll(val_logits / T.clamp_min(1e-3), val_labels)
    loss.backward()
    return loss

opt.step(_closure)
T_opt = float(T.detach().clamp_min(1e-3))

ece_raw = _ece_from_logits(test_logits, test_labels, T=1.0)
ece_cal = _ece_from_logits(test_logits, test_labels, T=T_opt)

print(f'Optimal temperature T = {T_opt:.3f}')
print(f'  ECE (raw)      : {ece_raw:.5f}')
print(f'  ECE (T-scaled) : {ece_cal:.5f}  [{"PASS" if ece_cal <= 0.05 else "FAIL"}]  target <= 0.05')

# Persist T next to the checkpoints; the dashboard Calibration tab loads this
# as its slider default so the same scaling is applied at inference time.
(CKPT_DIR / 'temperature.json').write_text(json.dumps({'temperature': T_opt}))
print(f'Saved -> {CKPT_DIR / "temperature.json"}')

### Metrics by data provenance — architecture validation vs field-relevant

The ingestion pipeline and the fallback generator write `data/provenance.json`, tagging each class's images as **real** (genuine annotated IR pixels), **remapped** (real FLIR ROI, intensity-remapped), or **synthetic** (procedurally generated). Until `label_maps.yaml` coverage improves, most classes are synthetic-only, so the headline accuracy is largely **architecture validation** — it shows the A→B→C pipeline learns and separates classes, not that it recognises real targets yet. This cell splits test accuracy by provenance so the two are never conflated.

In [ ]:
import json
from collections import defaultdict

PROV_PATH  = DATA_DIR / 'provenance.json'
provenance = json.loads(PROV_PATH.read_text()) if PROV_PATH.exists() else {}

def _prov_bucket(cls):
    """real_backed = any genuine/remapped IR pixels; else synthetic_only/unknown."""
    p     = provenance.get(cls, {})
    real  = p.get('real', 0) + p.get('remapped', 0)
    synth = p.get('synthetic', 0)
    if real == 0 and synth == 0:
        return 'unknown'
    return 'real_backed' if real > 0 else 'synthetic_only'

n_real  = sum(1 for c in CLASSES if _prov_bucket(c) == 'real_backed')
n_synth = sum(1 for c in CLASSES if _prov_bucket(c) == 'synthetic_only')
n_unk   = NUM_CLASSES - n_real - n_synth

if not provenance:
    print('No provenance.json found — re-run the ingestion + fallback cells to')
    print('generate it. Skipping provenance breakdown.')
else:
    # Split the test predictions (from c-eval-metrics) by the provenance
    # bucket of each sample's TRUE class.
    buck_correct = defaultdict(int)
    buck_total   = defaultdict(int)
    for pred, true in zip(all_preds, all_labels):
        b = _prov_bucket(idx_to_cls.get(true, ''))
        buck_correct[b] += int(pred == true)
        buck_total[b]   += 1

    print(f'Class provenance: {n_real} real-backed, {n_synth} synthetic-only, '
          f'{n_unk} unknown  (of {NUM_CLASSES})')
    print()
    print('Test accuracy by provenance of the true class:')
    for b in ['real_backed', 'synthetic_only', 'unknown']:
        t = buck_total.get(b, 0)
        if t:
            print(f'  {b:<15} {buck_correct[b] / t:.4f}  ({buck_correct[b]}/{t})')
    print()
    print('Interpretation:')
    print('  real_backed    → field-relevant signal (genuine/remapped IR pixels)')
    print('  synthetic_only → ARCHITECTURE VALIDATION only (procedural targets)')
    print(f'  Headline accuracy {test_acc:.4f} is dominated by the {n_synth} '
          f'synthetic-only classes;')
    print('  treat it as architecture validation until label-map coverage grows.')

    # Aggregate provenance composition across the whole dataset
    tot = defaultdict(int)
    for p in provenance.values():
        for k, v in p.items():
            tot[k] += v
    grand = sum(tot.values()) or 1
    print()
    print('Dataset composition (all images):')
    for k in ['real', 'remapped', 'synthetic']:
        print(f'  {k:<10} {tot.get(k, 0):>7}  ({100 * tot.get(k, 0) / grand:.1f}%)')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

cell_sz = max(0.4, 12 / NUM_CLASSES)
fig, ax = plt.subplots(figsize=(NUM_CLASSES * cell_sz + 2, NUM_CLASSES * cell_sz + 1))
sns.heatmap(
    cm,
    annot=(NUM_CLASSES <= 20),
    fmt='d',
    cmap='Blues',
    xticklabels=ordered_cls,
    yticklabels=ordered_cls,
    ax=ax,
    linewidths=0.3 if NUM_CLASSES <= 20 else 0,
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title(
    f'Confusion Matrix — Test Set  (acc={test_acc:.3f}, {NUM_CLASSES} classes)',
    fontsize=12,
)
plt.xticks(rotation=90, fontsize=max(5, 8 - NUM_CLASSES // 10))
plt.yticks(rotation=0,  fontsize=max(5, 8 - NUM_CLASSES // 10))
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

### Hard-negative mining (optional) — confusable-class fine-tune pass

If the confusion matrix above shows bleed within `{F16, MiG19, MiG21}` (fighter silhouettes),
`{BMP2, Bradley, K21}` (IFV/APC), or `{T72, T90, Leopard2}` (MBT), this cell mines the
misclassified / low-margin samples from those groups, oversamples them, and runs a short
low-LR fine-tune pass on top of the already-trained model — a post-hoc addition, not a
replacement for the 300-epoch schedule above. Skip if accuracy/confusion already look fine.

In [ ]:
from modules.hard_negative_mining import mine_hard_negatives, finetune_on_hard_negatives, CONFUSABLE_GROUPS
from modules.module_b_classifier import ARCHITECTURES

RUN_HARD_NEGATIVE_MINING = False   # <- set True to run this pass

if RUN_HARD_NEGATIVE_MINING:
    train_loader, _, _ = build_loaders(CONFIG)
    hn_model = ensemble.models[0] if USE_ENSEMBLE else eval_model
    hn_arch  = ARCHITECTURES[0] if USE_ENSEMBLE else 'convnext_tiny'

    hard_idx = mine_hard_negatives(hn_model, train_loader.dataset, device)
    print(f'Confusable groups: {CONFUSABLE_GROUPS}')
    print(f'{len(hard_idx)} hard negatives found out of {len(train_loader.dataset)} '
          f'training samples ({len(hard_idx) / max(len(train_loader.dataset), 1):.1%})')

    hn_ckpt = str(CKPT_DIR / 'hard_negative_finetuned.pth')
    best_hn_acc = finetune_on_hard_negatives(
        hn_model, train_loader.dataset, val_loader, device,
        arch=hn_arch, hard_indices=hard_idx,
        oversample_factor=4, epochs=15, lr=1e-5, ckpt_path=hn_ckpt,
    )
    print(f'Best val acc after hard-negative fine-tune: {best_hn_acc:.4f} -> {hn_ckpt}')
else:
    print('Hard-negative mining skipped (set RUN_HARD_NEGATIVE_MINING = True to run it).')

## Section 4 — Module C: XAI

- **Grad-CAM**: gradient-weighted class activation map — lights up the region that drove the decision
- **Faithfulness AUC**: progressively mask (deletion) or reveal (insertion) top-saliency pixels and measure class-probability drop/rise; AUC is the target metric (≥ 0.80)
- **MC Dropout**: Bayesian uncertainty estimate — high entropy = OOD / low-confidence flag

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

# GradCAM needs a plain ConvNeXt (not the ensemble wrapper)
cam_model = ensemble.models[0] if USE_ENSEMBLE else eval_model
cam_model.eval()
gradcam = GradCAMExplainer(cam_model)

# Gather up to 3 test samples per domain
samples_by_dom = {'AIR': [], 'GROUND': [], 'NAVAL': []}
with torch.no_grad():
    for imgs, labels in test_loader:
        for img, lbl in zip(imgs, labels):
            cls_id = idx_to_cls.get(lbl.item(), '')
            dom    = TARGET_META.get(cls_id, {}).get('domain', '')
            if dom in samples_by_dom and len(samples_by_dom[dom]) < 3:
                samples_by_dom[dom].append((img, lbl.item(), cls_id))
        if all(len(v) >= 3 for v in samples_by_dom.values()):
            break

fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for row, dom in enumerate(['AIR', 'GROUND', 'NAVAL']):
    for col, (img, lbl_idx, cls_id) in enumerate(samples_by_dom[dom]):
        x    = img.unsqueeze(0).to(device)
        pred = cam_model(x).argmax(1).item()
        hmap = gradcam.explain(x, pred)

        orig = (img.permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
        gray = (orig[:, :, 0] * 255).clip(0, 255).astype(np.uint8)
        gray_rgb = np.stack([gray] * 3, -1)
        heat_rgb = cv2.applyColorMap((hmap * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
        over = cv2.addWeighted(gray_rgb, 0.55, heat_rgb, 0.45, 0)

        ax_raw = axes[row, col * 2]
        ax_cam = axes[row, col * 2 + 1]
        ax_raw.imshow(gray, cmap='gray')
        ax_raw.set_title(f'True: {cls_id}', fontsize=8)
        ax_raw.axis('off')
        pred_name = idx_to_cls.get(pred, str(pred))
        color = 'green' if pred == lbl_idx else 'red'
        ax_cam.imshow(over)
        ax_cam.set_title(f'Pred: {pred_name}', fontsize=8, color=color)
        ax_cam.axis('off')
    axes[row, 0].set_ylabel(dom, fontsize=12, fontweight='bold', labelpad=40)

plt.suptitle('Grad-CAM  |  left: input  right: attention heatmap  (green=correct, red=wrong)',
             fontsize=11)
plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_by_domain.png', dpi=150)
plt.show()
print('Saved: gradcam_by_domain.png')

In [ ]:
import importlib, numpy as np, time, sys

# ── Force-reload module_c_xai and rebuild gradcam ──────────────────────
# Kaggle keeps the old GradCAMExplainer instance in memory even after
# git pull + sys.modules wipe; this guarantees we use the latest class.
for _m in [k for k in list(sys.modules) if 'module_c_xai' in k]:
    del sys.modules[_m]
import modules.module_c_xai as _xai
importlib.reload(_xai)
from modules.module_c_xai import (
    GradCAMExplainer, faithfulness_deletion_auc, faithfulness_insertion_auc,
)

faith_model = ensemble.models[0] if USE_ENSEMBLE else eval_model
faith_model.eval()
for _p in faith_model.parameters():
    _p.requires_grad_(True)

gradcam = GradCAMExplainer(faith_model)   # rebuilt with the patched class

del_aucs, ins_aucs = [], []
N_FAITH = 30   # enough for stable estimate; each call takes ~0.5 s on T4

t0 = time.time()
collected = 0
for imgs, labels in test_loader:
    for img, lbl in zip(imgs[:4], labels[:4]):
        if collected >= N_FAITH:
            break
        x = img.unsqueeze(0).to(device)
        with torch.no_grad():
            pred = faith_model(x).argmax(1).item()
        sal = gradcam.explain(x, pred)
        del_aucs.append(faithfulness_deletion_auc(faith_model, x, sal, pred,
                                                   steps=20, device=device))
        ins_aucs.append(faithfulness_insertion_auc(faith_model, x, sal, pred,
                                                    steps=20, device=device))
        collected += 1
    if collected >= N_FAITH:
        break

mean_del = float(np.mean(del_aucs))
mean_ins = float(np.mean(ins_aucs))
elapsed  = time.time() - t0

print(f'Faithfulness Deletion  AUC : {mean_del:.4f}  [{"PASS" if mean_del >= 0.80 else "FAIL"}]  target >= 0.80')
print(f'Faithfulness Insertion AUC : {mean_ins:.4f}  [{"PASS" if mean_ins >= 0.80 else "FAIL"}]  target >= 0.80')
print(f'n={N_FAITH} samples, {elapsed:.0f} s')

In [ ]:
# MC Dropout uncertainty: run N stochastic forward passes
mc_wrapper = MCDropoutWrapper(cam_model, n_samples=20)

imgs, labels = next(iter(test_loader))
sample = imgs[:6].to(device)

result = mc_wrapper(sample)
mean_probs  = result['mean_probs'].cpu().numpy()   # (6, C)
uncertainty = result['uncertainty'].cpu().numpy()  # (6,)  entropy

print('MC Dropout uncertainty (entropy) per sample:')
print(f'{"#":<4}  {"Class":<14}  {"Conf":>6}  {"Entropy":>8}  {"OOD?"}')
OOD_THRESH = 2.0   # heuristic: log(43) * 0.5 ~ 1.9
for i in range(6):
    pred_idx  = int(mean_probs[i].argmax())
    pred_name = ordered_cls[pred_idx] if pred_idx < len(ordered_cls) else str(pred_idx)
    conf      = mean_probs[i, pred_idx]
    ent       = uncertainty[i]
    flag      = 'OOD' if ent > OOD_THRESH else 'OK'
    print(f'{i:<4}  {pred_name:<14}  {conf:>6.3f}  {ent:>8.3f}  {flag}')

## Section 5 — End-to-End Pipeline Demo (A → B → C)

Shows the full inference pipeline on a synthetic IR frame:
1. **Module A** (`IRDetector.detect`) — YOLOv4 forward pass + NMS → bounding boxes
2. **Module B** — ConvNeXt forward pass on each ROI crop → class + confidence
3. **Module C** — Grad-CAM heatmap per detection

The bootstrap cell below initialises Module A from the official COCO YOLOv4 darknet weights (backbone + neck; ~256 MB one-time download). This is **not** IR-trained — detections become meaningful only after fine-tuning on a labelled IR detection set — but it beats random init and makes the dashboard's default checkpoint path work end-to-end.

Note the fail-loud contract: `IRDetector(weights=path)` raises `FileNotFoundError` on a missing path; `weights=None` is the explicit untrained fallback.

In [ ]:
import shutil, subprocess

# ── Bootstrap Module A detector weights ────────────────────────────────
# Downloads official COCO YOLOv4 darknet weights and converts them to a
# REATS checkpoint (backbone + neck; heads stay random until IR training).
BOOTSTRAP = REATS / 'bootstrap_detector_weights.py'
DET_CKPT  = REATS / 'checkpoints' / 'detector_bootstrap.pt'

if DET_CKPT.exists():
    print(f'Already bootstrapped: {DET_CKPT}  ({DET_CKPT.stat().st_size / 1e6:.1f} MB)')
else:
    r = subprocess.run(['python', str(BOOTSTRAP)],
                       capture_output=True, text=True, cwd=str(REATS))
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-800:])

# Mirror into /kaggle/working/checkpoints so the dashboard's default
# sidebar path (checkpoints/detector_bootstrap.pt, resolved relative to
# the streamlit CWD = /kaggle/working) loads without editing.
if DET_CKPT.exists():
    shutil.copy2(DET_CKPT, CKPT_DIR / 'detector_bootstrap.pt')
    print(f'Mirrored -> {CKPT_DIR / "detector_bootstrap.pt"}')
else:
    print('Bootstrap failed — c-module-a will fall back to weights=None (untrained).')

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from modules.module_a_detector import IRDetector

# Synthetic 640x512 IR frame with two hot blobs (stand-ins for real FLIR video)
def make_ir_frame(h: int = 512, w: int = 640) -> np.ndarray:
    rng   = np.random.default_rng(0)
    frame = (rng.normal(loc=80, scale=10, size=(h, w))).clip(0, 255).astype(np.float32)
    for cy, cx, r, bright in [(160, 200, 30, 230), (380, 460, 20, 215)]:
        Y, X  = np.ogrid[:h, :w]
        dist  = np.sqrt((Y - cy) ** 2 + (X - cx) ** 2)
        mask  = dist < r
        frame[mask] = bright * (1 - dist[mask] / r)
    frame = frame.clip(0, 255).astype(np.uint8)
    return cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)

# Initialise Module A from the bootstrap checkpoint when present.
# IRDetector raises FileNotFoundError on an explicit missing path, so we
# guard here; weights=None is the explicit untrained fallback.
DET_CKPT = REATS / 'checkpoints' / 'detector_bootstrap.pt'
detector = IRDetector(
    weights=str(DET_CKPT) if DET_CKPT.exists() else None,
    num_classes=NUM_CLASSES, conf=0.25, iou=0.45,
)
print('Module A weights:', detector.weights_path
      or 'random init (untrained — run the bootstrap cell above first)')

test_frame = make_ir_frame()
detections = detector.detect(test_frame)

print(f'Module A: {len(detections)} detection(s) on synthetic frame')
for det in detections:
    cls_id = CLASSES[det['class_id']]
    dom    = TARGET_META.get(cls_id, {}).get('domain', '?')
    print(f'  {cls_id:<14}  domain={dom}  conf={det["conf"]:.3f}  bbox={det["bbox"]}')

# Visualise frame (annotated if detections exist)
vis = test_frame.copy()
for det in detections:
    x1, y1, x2, y2 = [int(v) for v in det['bbox']]
    cls_id = CLASSES[det['class_id']]
    color  = tuple(int(c) for c in TARGET_META.get(cls_id, {}).get('color_bgr', [0, 255, 0]))
    cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
    cv2.putText(vis, f'{cls_id} {det["conf"]:.2f}', (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(test_frame[:, :, 0], cmap='inferno')
axes[0].set_title('Synthetic IR Frame', fontsize=11)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Module A — {len(detections)} detection(s)', fontsize=11)
axes[1].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/module_a_demo.png', dpi=150)
plt.show()

In [ ]:
import time
from torchvision import transforms

roi_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])


def reats_pipeline(frame_bgr: np.ndarray, top_k: int = 3):
    """
    A -> B -> C pipeline for one IR frame.
    Returns list of result dicts and latency_ms.
    """
    t0 = time.perf_counter()

    dets = detector.detect(frame_bgr)
    if not dets:
        # Treat the whole frame as one ROI when Module A has no detections yet
        h, w = frame_bgr.shape[:2]
        dets = [{'bbox': [0, 0, w, h], 'conf': 1.0, 'class_id': 0}]

    results = []
    bm = ensemble.models[0] if USE_ENSEMBLE else eval_model
    bm.eval()
    cam = GradCAMExplainer(bm)

    for det in dets:
        x1, y1, x2, y2 = [int(v) for v in det['bbox']]
        roi = frame_bgr[max(0, y1):y2, max(0, x1):x2]
        if roi.size == 0:
            continue
        x     = roi_transform(roi).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = bm(x)
            probs  = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        top_idx   = probs.argsort()[::-1][:top_k]
        pred_cls  = ordered_cls[top_idx[0]] if top_idx[0] < len(ordered_cls) else CLASSES[top_idx[0]]
        heatmap   = cam.explain(x, int(top_idx[0]))
        results.append({
            'bbox':       det['bbox'],
            'class_name': pred_cls,
            'confidence': float(probs[top_idx[0]]),
            'domain':     TARGET_META.get(pred_cls, {}).get('domain', '?'),
            'top_k':      [(ordered_cls[i] if i < len(ordered_cls) else str(i),
                            float(probs[i])) for i in top_idx],
            'heatmap':    heatmap,
            'roi':        roi,
        })

    latency_ms = (time.perf_counter() - t0) * 1000
    return results, latency_ms


# Warm-up call (untimed, discarded) — excludes one-time costs (CUDA kernel
# compilation, first GradCAMExplainer hook registration/module scan) from the
# reported number. The 2026-07-04 Kaggle run's 111.9ms end-to-end latency was
# exactly this kind of cold-path measurement — flagged in the run report as
# "not representative of steady-state" and not comparable to the <=40ms target.
_ = reats_pipeline(test_frame)

# Now run (and time) on the warm path, averaged over a few frames for stability
N_LATENCY_REPS = 5
_lat_samples = []
for _ in range(N_LATENCY_REPS):
    pipe_results, _lat = reats_pipeline(test_frame)
    _lat_samples.append(_lat)
latency_ms = float(np.mean(_lat_samples))

lat_ok = latency_ms <= 40
print(f'End-to-end latency (warm, mean of {N_LATENCY_REPS}): {latency_ms:.1f} ms  '
      f'[{"PASS" if lat_ok else "OVER TARGET"}]  (target <= 40 ms)')
print(f'  per-rep: {[round(v, 1) for v in _lat_samples]} ms')
print()
for r in pipe_results:
    print(f'  [{r["domain"]}]  {r["class_name"]}  conf={r["confidence"]:.3f}')
    for cls_name, p in r['top_k']:
        print(f'      {cls_name:<15}  {p:.3f}')

# Visualise result panel
n_res = len(pipe_results)
if n_res > 0:
    fig, axes = plt.subplots(n_res, 3, figsize=(12, 3.5 * n_res),
                             squeeze=False)
    for i, r in enumerate(pipe_results):
        roi_gray = r['roi'][:, :, 0] if r['roi'].ndim == 3 else r['roi']
        hmap_col = cv2.applyColorMap(
            (r['heatmap'] * 255).astype(np.uint8), cv2.COLORMAP_JET
        )[..., ::-1]
        h224, w224 = 224, 224
        roi_rgb  = cv2.resize(np.stack([roi_gray] * 3, -1), (w224, h224))
        hmap_res = cv2.resize(hmap_col, (w224, h224))
        overlay  = cv2.addWeighted(roi_rgb, 0.55, hmap_res, 0.45, 0)

        bar_vals   = [p for _, p in r['top_k']]
        bar_labels = [n for n, _ in r['top_k']]

        axes[i, 0].imshow(roi_gray, cmap='gray')
        axes[i, 0].set_title('ROI crop', fontsize=9)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title(f'Grad-CAM  [{r["class_name"]}  {r["confidence"]:.2f}]', fontsize=9)
        axes[i, 1].axis('off')

        axes[i, 2].barh(bar_labels[::-1], bar_vals[::-1], color='#1976D2')
        axes[i, 2].set_xlim(0, 1)
        axes[i, 2].set_title(f'Top-{len(bar_vals)} probs', fontsize=9)
        axes[i, 2].tick_params(labelsize=8)

    plt.suptitle(f'A->B->C Pipeline  ({latency_ms:.0f} ms, warm)', fontsize=12)
    plt.tight_layout()
    plt.savefig('/kaggle/working/pipeline_demo.png', dpi=150)
    plt.show()
    print('Saved: pipeline_demo.png')

## Section 6 — Module D: Streamlit Dashboard

The dashboard has 5 tabs:
- **Live Analysis** — upload a frame or FLIR video; full A→B→C pipeline; annotated output with threat-level colour overlay
- **Batch** — run on a folder of frames; export results CSV
- **Calibration** — temperature scaling; ECE + reliability diagram
- **About** — 43-class taxonomy grouped by domain + threat level legend
- **iPhone Live Feed** — live MJPEG stream from Module E (FastAPI WebSocket server); paste the ngrok/Cloudflare URL from Module E into the URL field

To serve it from Kaggle, expose port 8501 via an ngrok tunnel.

In [ ]:
import subprocess, sys, time, socket
from pathlib import Path

ROOT  = Path('/kaggle/working/reats')
REATS = ROOT / 'REATS'

# ── Pre-flight: the dashboard fails loudly on missing checkpoints ──────
# Sidebar defaults resolve relative to the streamlit CWD (/kaggle/working):
#   checkpoints/detector_bootstrap.pt   (from the bootstrap cell)
#   checkpoints/convnext_best.pth       (from c-train-single)
CKPT_DIR = Path('/kaggle/working/checkpoints')
print('Checkpoint pre-flight:')
for _name in ['detector_bootstrap.pt', 'convnext_best.pth']:
    _p = CKPT_DIR / _name
    print(f'  {"OK     " if _p.exists() else "MISSING"}  checkpoints/{_name}')
print()

# ── Set your ngrok token (free account at https://ngrok.com) ───────────
# Get yours at: https://dashboard.ngrok.com/get-started/your-authtoken
# Or use Kaggle Secrets (Add-ons → Secrets) and:
#   from kaggle_secrets import UserSecretsClient
#   NGROK_TOKEN = UserSecretsClient().get_secret('NGROK_AUTHTOKEN')
NGROK_TOKEN = ''   # <- paste token here

if NGROK_TOKEN:
    # ── Ensure streamlit is installed ───────────────────────────────────
    try:
        import streamlit  # noqa: F401
    except ImportError:
        print('Installing streamlit...')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            'streamlit>=1.30.0', 'watchdog>=3.0.0',
        ])

    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN

    # Kill any old streamlit / ngrok processes from a previous run
    ngrok.kill()
    subprocess.run(['pkill', '-f', 'streamlit'], check=False)
    time.sleep(1)

    dashboard_path = REATS / 'modules' / 'module_d_dashboard.py'
    if not dashboard_path.exists():
        raise FileNotFoundError(
            f'{dashboard_path} not found. Run cell 3 (c-clone) first.'
        )

    # Launch streamlit, capture its stderr so we can show errors if it dies
    log_path = Path('/kaggle/working/streamlit.log')
    log_fp   = open(log_path, 'wb')
    proc     = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', str(dashboard_path),
         '--server.port', '8501',
         '--server.headless', 'true',
         '--server.address', '0.0.0.0',
         '--server.fileWatcherType', 'none',
         '--browser.gatherUsageStats', 'false'],
        stdout=log_fp, stderr=subprocess.STDOUT,
    )

    # Wait for port 8501 to actually accept connections before opening tunnel
    def _port_open(port: int) -> bool:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(0.5)
            return s.connect_ex(('127.0.0.1', port)) == 0

    print('Waiting for streamlit to start...')
    for i in range(60):                       # up to 60 s
        if proc.poll() is not None:
            print('streamlit died — last 40 log lines:')
            print(log_path.read_text().splitlines()[-40:])
            raise RuntimeError('streamlit failed to start')
        if _port_open(8501):
            print(f'  ready in {i+1}s')
            break
        time.sleep(1)
    else:
        raise TimeoutError('streamlit did not open port 8501 within 60 s')

    tunnel = ngrok.connect(8501, 'http')
    print(f'Dashboard: {tunnel.public_url}')
    print('Open the URL above in a new browser tab.')
else:
    print('Set NGROK_TOKEN above to launch the live dashboard.')

In [ ]:
from pathlib import Path

print('=' * 58)
print('  REATS Artifacts — /kaggle/working/')
print('=' * 58)
ARTIFACT_EXTS = {'.pth', '.pt', '.png', '.yaml', '.json', '.db', '.csv'}
for p in sorted(Path('/kaggle/working').rglob('*')):
    if p.is_file() and p.suffix in ARTIFACT_EXTS:
        size_kb = p.stat().st_size / 1024
        rel = str(p.relative_to('/kaggle/working'))
        print(f'  {rel:<50}  {size_kb:7.1f} KB')
print('=' * 58)
print()
print('IMPORTANT: /kaggle/working/ is wiped when the session ends.')
print('Download these before closing:')
print('  checkpoints/convnext_best.pth               <- single model')
print('  checkpoints/{arch}_{i}.pth                   <- ensemble (if trained;')
print('                                                   arch in ARCHITECTURES,')
print('                                                   e.g. convnext_tiny_0.pth,')
print('                                                   resnext50_1.pth, ...)')
print('  checkpoints/hard_negative_finetuned.pth      <- if hard-neg cell was run')
print('  checkpoints/detector_bootstrap.pt            <- Module A bootstrap')
print('  checkpoints/temperature.json                 <- calibration T')
print()
print('Performance summary:')
print(f'  Accuracy      {test_acc:.4f}  [{"PASS" if test_acc >= 0.92 else "FAIL"}]  target >= 0.92')
if 'ece_cal' in dir():
    print(f'  ECE (T={T_opt:.2f})  {ece_cal:.5f}  [{"PASS" if ece_cal <= 0.05 else "FAIL"}]  target <= 0.05')
else:
    print(f'  ECE           {ece:.5f}  [{"PASS" if ece <= 0.05 else "FAIL"}]  target <= 0.05')
if 'far_mr' in dir():
    print(f'  RED FAR/MR    {far_mr["red_threat_FAR"]:.4f} / {far_mr["red_threat_MR"]:.4f}  (no pass/fail target)')
if 'mean_del' in dir():
    print(f'  Faith. AUC    {mean_del:.4f}  [{"PASS" if mean_del >= 0.80 else "FAIL"}]  target >= 0.80')
if 'latency_ms' in dir():
    print(f'  Latency       {latency_ms:.1f} ms  [{"PASS" if latency_ms <= 40 else "OVER"}]  target <= 40 ms')